# AC power estimation

<table>
  <tr>
    <td><img src="https://raw.githubusercontent.com/Mohamed-ENNHIRI/ENER958/main/notebooks/logos/solaracademy.png" width="200" />
    <td><img src="https://raw.githubusercontent.com/Mohamed-ENNHIRI/ENER958/main/notebooks/logos/usmb.png" width="200" />
    <td><img src="https://raw.githubusercontent.com/Mohamed-ENNHIRI/ENER958/main/notebooks/logos/cnrs.png" width="80" />
  </tr>
</table>

<sub>Author: Mohamed Ennhiri, PhD student, LOCIE UMR CNRS 5271, Université Savoie Mont Blanc</sub>

<sub>Structure and format inspired by the course given by Alexandre Mathieu at the Solar Academy in 2024, "PV failure detection based on operational time series". Physics and data: pvlib, PVGIS, ODRE, RTE.</sub>

This notebook introduces some tools to estimate the AC power production of a PV installation and to assess its performance.

It is broken down into three parts:

1. **Data visualization**: Exploration of the weather data and of the operating data of the system.
2. **POA irradiance, temperature and power modeling**: Estimation of the POA irradiance, the module temperature, the DC and AC power, and comparison to the actual values.
3. **Performance metrics**: Performance ratio, energy performance index, and accuracy of the models.

This notebook is the rehearsal of part A of the project, on the course site.

The students are expected to do the **exercices**.

Data: course site, Savoie, near Le Bourget-du-Lac (Lat 45.64 N, Long 5.92 E). Weather from PVGIS SARAH3, 2023.

##### Characteristics of the PV installation:
- tilt: 30°
- azimuth: 180° (south)
- albedo: 0.2 (assumption)
- nominal DC power: 3 kWp
- inverter AC rating: 2.5 kW

##### Module datasheet
- crystalline silicon
- thermal (power) coefficient: −0.35 %/°C
- NOCT: 45 °C

##### Generic temperature coefficients for an open-rack installation (Faiman)
- u_0 = 25.0 W/(°C.m²)
- u_1 = 6.84 W.s/(°C.m³)

## I. Data visualization

In [ ]:
# Please execute this cell to import packages
import numpy as np # For math operations
import pandas as pd # To create indexed time series
import matplotlib.pyplot as plt # To plot figures

# In Google Colab, pvlib is not installed by default: the line below installs it (takes 30 s)
!pip install -q pvlib
import pvlib
print("pvlib", pvlib.__version__)

### I.1 Weather data

Below are the weather inputs of the models. Execute the cell below to import them !

In [ ]:
# Load the satellite weather data of the course site from an online file: GHI, DNI, DHI, air temperature, wind speed
# Site: Savoie, near Le Bourget-du-Lac (Lat 45.64 N, Long 5.92 E). Source: PVGIS SARAH3, year 2023, hourly.
urlw = "https://raw.githubusercontent.com/Mohamed-ENNHIRI/ENER958/main/data/weather_45.64N_5.92E_2023.csv"
weather_data = pd.read_csv(urlw, index_col=0) # Import the csv file and use the first column as index
weather_data.index = pd.to_datetime(weather_data.index) # Convert the index into datetimes (UTC)
print(weather_data.head(5)) # Show the first five lines

In [ ]:
# Load the solar position (zenith, azimuth, elevation) calculated with the NREL algorithm:
# I. Reda and A. Andreas, Solar position algorithm for solar radiation applications. Solar Energy, 76(5), 2004.
urls = "https://raw.githubusercontent.com/Mohamed-ENNHIRI/ENER958/main/data/solarpos_45.64N_5.92E_2023.csv"
solar_position = pd.read_csv(urls, index_col=0)
solar_position.index = pd.to_datetime(solar_position.index)
print(solar_position.head(5))

Visualize the data with the plot below ! (Execute the cell)

In [ ]:
# Make a figure with two axes (two figures top / bottom) which share the same x axis
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
weather_data["temp_air"].plot(ax=ax1, title="Air temperature [°C] as function of time", ylabel="Temperature [°C]")
weather_data[["ghi", "dni", "dhi"]].plot(ax=ax2, title="Irradiance [W/m²] as function of time", ylabel="Irradiance [W/m²]")
plt.tight_layout()

### I.2 Operating data of the system

Actuals are imported by executing the cell below.

- g_poa: global POA irradiance [W/m²]
- t_mod: module temperature [°C]
- p_dc: DC power [W]
- p_ac: AC power [W]

In [ ]:
# Load the actual data of the system from an online file
url = "https://raw.githubusercontent.com/Mohamed-ENNHIRI/ENER958/main/data/actuals_course_site_2023.csv"
actual = pd.read_csv(url, index_col=0)
actual.index = pd.to_datetime(actual.index)
print(actual.head(10))

#### Exercice

Plot the DC and AC power of the actuals over the first week of July on one axis, and the module temperature on a second axis sharing the x axis.

In [ ]:
# Write your code here


## II. AC power estimation

Modeling PV power from ghi / dhi / dni goes into four steps:

1. Modeling the POA irradiance.
2. Modeling the module cell temperature.
3. Modeling the DC power.
4. Modeling the inverter efficiency and the AC power.

### II.1 POA irradiance

#### Exercice

Calculate the POA irradiance with the isotropic assumption, as introduced in the notebook `python_intro_poa`, and compare with `actual["g_poa"]` over one week.

In [ ]:
# Write your code here


### II.2 Cell temperature model

#### Exercice

Calculate the module temperature with, both, the NOCT and the Faiman model, and compare with `actual["t_mod"]` over four days.

In [ ]:
NOCT = 45 # From the datasheet

# Generic coefficients for an open-rack PV installation
u_0 = 25.0 # W/(°C.m2)
u_1 = 6.84 # W.s/(°C.m3)

# Write your code here


### II.3 DC power model

#### Exercice

Calculate the DC power with the PVWatts model (write a function) with both temperature models.

In [ ]:
Pdc_0 = 3000 # Wp, nominal power
gamma_pdc = -0.35 / 100 # [-]/°C

# Write your code here


### II.4 AC power model

The inverter efficiency depends on the load. The PVWatts inverter model:

$\eta = \frac{\eta_{nom}}{\eta_{ref}} (-0.0162 \zeta - \frac{0.0059}{\zeta} + 0.9858)$ with $\zeta = P_{DC} / P_{DC,0}$, $\eta_{nom}$ = 0.96 and $\eta_{ref}$ = 0.9637.

The AC power is $P_{AC} = \eta \cdot P_{DC}$, capped at the AC rating of the inverter $P_{AC,0}$.

#### Exercice

Calculate the AC power with the PVWatts inverter model (write a function). Then compare the estimated AC power with the actual AC power over one week.

In [ ]:
Pac_0 = 2500 # W, AC rating of the inverter

# Write your code here


## III. Performance metrics

In this section, the performance of the installation and the accuracy of the models are assessed.

- Performance ratio: $PR = \frac{E_{AC}}{P_0 \cdot H_{POA} / 1000}$, with $H_{POA}$ the annual POA irradiation in Wh/m²
- Energy performance index: $EPI = \frac{E_{AC, actual}}{E_{AC, estimated}}$

#### Exercice

Calculate the PR of the actual system and the PR expected from the model, then the EPI.

In [ ]:
# Write your code here


#### Exercice, to go further

What is the accuracy of the estimated values compared to the actual ones ? Calculate the mean bias error (MBE), the mean absolute error (MAE) and the root mean square error (RMSE) of the POA irradiance, the module temperature and the AC power, for the daytime hours (g_poa > 10 W/m²). Which variable carries the largest relative error?

In [ ]:
# Write your code here


#### Exercice, to go further

The actual AC power is zero on three days of the year while the model is not. Find these days (daily actual energy below 1 % of the modelled one) and print the energy lost, in kWh. This is an availability loss, the first "fault" of the project.

In [ ]:
# Write your code here
